# ML Practical 2 – Linear, Ridge and Lasso Regression

**Aim:** Analyze the Uber fare dataset, clean and explore it, then compare Linear, Ridge and Lasso regression models.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file_name = "uber.csv"

df = pd.read_csv(file_name)
print(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
print("\nDataset Information:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())

# Fill missing numerical values with median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

# Convert datetime and remove non-predictive columns
if "pickup_datetime" in df.columns:
    df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce", utc=True)

drop_cols = ["key", "Unnamed: 0", "pickup_datetime"]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
df.dropna(inplace=True)

print("\nCleaned data:")
print(df.head())
print(df.isnull().sum())

In [ ]:
corr = df.corr(numeric_only=True)
print(corr)

plt.figure(figsize=(10, 7))
plt.imshow(corr, cmap="coolwarm", aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
df.boxplot()
plt.title("Boxplot for Outlier Detection")
plt.xticks(rotation=45)
plt.show()

In [ ]:
X = df.drop("fare_amount", axis=1)
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)
linear_pred = linear_model.predict(X_test_scaled)

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
ridge_pred = ridge_model.predict(X_test_scaled)

lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_train)
lasso_pred = lasso_model.predict(X_test_scaled)

print("All three regression models trained.")

In [ ]:
def evaluate_model(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    print("\n", name)
    print("-" * 30)
    print("MAE:", mean_absolute_error(y_true, y_pred))
    print("MSE:", mse)
    print("RMSE:", np.sqrt(mse))
    print("R2:", r2_score(y_true, y_pred))

evaluate_model("Linear Regression", y_test, linear_pred)
evaluate_model("Ridge Regression", y_test, ridge_pred)
evaluate_model("Lasso Regression", y_test, lasso_pred)

results = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression", "Lasso Regression"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, linear_pred)),
        np.sqrt(mean_squared_error(y_test, ridge_pred)),
        np.sqrt(mean_squared_error(y_test, lasso_pred))
    ],
    "R2 Score": [
        r2_score(y_test, linear_pred),
        r2_score(y_test, ridge_pred),
        r2_score(y_test, lasso_pred)
    ]
})
print(results)

In [ ]:
# ===== ML PRACTICAL 2: LINEAR, RIDGE & LASSO =====

# AIM: Predict Uber fares and compare 3 regression models.
# FLOW: Load -> Clean -> Analyze -> Split -> Scale -> Train -> Evaluate

# THEORY:
# Regression = predicts continuous numerical values.
# Linear Regression = best-fit line (y = mx + c).
# Gradient Descent = updates weights to minimize error.
# Cost Function = measures prediction error (e.g., MSE).
# Linear = no penalty; Ridge = L2; Lasso = L1.
# Ridge shrinks weights; Lasso can make weights zero.
# alpha = regularization strength (higher = stronger penalty).

# DATA LOADING:
# read_csv() = loads CSV; head() = first 5 rows.
# shape = rows, columns; info() = dataset information.
# columns.tolist() = column names as list.
# isnull().sum() = missing value counts.

# CLEANING:
# select_dtypes(include=np.number) = numerical columns.
# median() = middle value; fillna() = fills missing values.
# to_datetime() = converts dates; errors="coerce" = invalid -> NaT.
# utc=True = UTC timezone.
# drop(columns=...) = removes columns; dropna() = removes missing rows.
# inplace=True = modifies original DataFrame.

# ANALYSIS:
# corr(numeric_only=True) = numerical correlation (-1 to +1).
# imshow(corr, cmap="coolwarm", aspect="auto") = heatmap.
# colorbar() = color scale; xticks()/yticks() = axis labels.
# boxplot() = detects possible outliers (doesn't remove them).
# figure(figsize=(10,7)) = graph size in inches.
# title() = graph title; show() = displays graph.
# tight_layout() = adjusts spacing; rotation=90 = rotates labels.

# SPLITTING & SCALING:
# X = inputs; y = fare_amount (target).
# drop("fare_amount", axis=1) = removes target column.
# train_test_split(X,y,test_size=0.2,random_state=42)
# = 80% training, 20% testing; 42 = reproducible split.
# StandardScaler() = standardizes data (mean 0, std 1).
# fit_transform(X_train) = learns scaling + transforms training data.
# transform(X_test) = scales testing data without data leakage.

# MODEL TRAINING:
# LinearRegression() = normal regression.
# Ridge(alpha=1.0) = L2 regularization.
# Lasso(alpha=0.1) = L1 regularization.
# fit(X_train,y_train) = trains model.
# predict(X_test) = predicts fares.

# EVALUATION:
# def evaluate_model(name,y_true,y_pred) = custom evaluation function.
# MAE = average absolute error (lower better).
# MSE = average squared error (lower better).
# RMSE = sqrt(MSE) (lower better).
# R2 = model performance score (higher better, max 1).
# mean_absolute_error() = MAE; mean_squared_error() = MSE.
# np.sqrt() = square root; r2_score() = R2.
# pd.DataFrame({...}) = creates comparison table.
# BEST MODEL = lowest test RMSE and highest test R2.

# ===== 6 MOST IMPORTANT VIVA QUESTIONS =====

# Q1. What is Linear Regression?
# Ans: Supervised algorithm predicting values using a best-fit line.

# Q2. What is Gradient Descent?
# Ans: Algorithm that updates weights to minimize cost.

# Q3. What is Cost Function?
# Ans: Measures prediction error, e.g., MSE.

# Q4. Difference between Linear, Ridge and Lasso?
# Ans: Linear = no penalty; Ridge = L2; Lasso = L1.
# Ridge shrinks weights; Lasso can remove features.

# Q5. Why use StandardScaler and train_test_split?
# Ans: Scaler standardizes features; split tests on unseen data.

# Q6. How do you select the best regression model?
# Ans: Lowest test RMSE and highest test R2.

# FINAL EXPLANATION:
# "We load and clean Uber data, analyze correlation and outliers,
# split 80:20, scale features, train Linear/Ridge/Lasso models,